In [20]:
#Importando dados
import pandas as pd
import numpy as np
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Treino:", train.shape)
print("Teste:", test.shape)

display(train.head())
train.info()

Treino: (1460, 81)
Teste: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [21]:
#Verificar valores ausentes
valores_ausentes = train.isnull().sum()

valores_ausentes = valores_ausentes[
    valores_ausentes > 0
].sort_values(ascending=False)

percentual_ausentes = (
    valores_ausentes / len(train) * 100
)

tabela_ausentes = pd.DataFrame({
    "Quantidade": valores_ausentes,
    "Percentual": percentual_ausentes
})

display(tabela_ausentes)

,Quantidade,Percentual
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


In [22]:
# Separar variavel alvo
y = train["SalePrice"]

X_train = train.drop(columns=["SalePrice"])
X_test = test.copy()

#Juntando treino e teste
dados = pd.concat(
    [X_train, X_test],
    axis=0,
    ignore_index=True
)

print(dados.shape)

#Removendo coluna ID
test_ids = X_test["Id"].copy()

dados = dados.drop(columns=["Id"])


(2919, 80)


In [23]:
#Tratando ausências
colunas_ausencia = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType"
]

for coluna in colunas_ausencia:
    if coluna in dados.columns:
        dados[coluna] = dados[coluna].fillna("None")

#Preenchendo valores ausentes
colunas_numericas_zero = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "MasVnrArea"
]

for coluna in colunas_numericas_zero:
    if coluna in dados.columns:
        dados[coluna] = dados[coluna].fillna(0)

#Preenchendo variaveis categoricas
colunas_categoricas = dados.select_dtypes(
    include=["object", "category"]
).columns

for coluna in colunas_categoricas:
    if dados[coluna].isnull().sum() > 0:
        dados[coluna] = dados[coluna].fillna(
            dados[coluna].mode()[0]
        )

#Preenchendo outras variaveis numericas
colunas_numericas = dados.select_dtypes(
    include=np.number
).columns

for coluna in colunas_numericas:
    if dados[coluna].isnull().sum() > 0:
        dados[coluna] = dados[coluna].fillna(
            dados[coluna].median()
        )

#Corrigindo classificação de categorias
colunas_para_texto = [
    "MSSubClass",
    "MoSold",
    "YrSold"
]

for coluna in colunas_para_texto:
    if coluna in dados.columns:
        dados[coluna] = dados[coluna].astype(str)

#Separando teste e treino novamente
n_train = len(X_train)

X_limpo = dados.iloc[:n_train].copy()
test_limpo = dados.iloc[n_train:].copy()

print(X_limpo.shape)
print(test_limpo.shape)

#Retirando os IDs como variaveis do modelo
# Pegue os IDs diretamente do arquivo original de teste
test_ids = test["Id"].copy()

# Remova Id apenas se ainda existir
X_limpo = X_limpo.drop(columns=["Id"], errors="ignore")
test_limpo = test_limpo.drop(columns=["Id"], errors="ignore")

print(X_limpo.shape)
print(test_limpo.shape)
print(test_ids.shape)

(1460, 79)
(1459, 79)
(1460, 79)
(1459, 79)
(1459,)
